# Quick check for `create_dfs`
This notebook shows a minimal end-to-end usage of `create_train_df` and `create_inference_df`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.wait_time_forecasting.paths import notebook_context
from modeling.wait_time_forecasting import create_train_df, create_inference_df

ctx = notebook_context(project_root=PROJECT_ROOT)
PROJECT_ROOT = ctx["project_root"]
DATA_DIR = ctx["data_dir"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code
DATA_DIR: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\raw


In [2]:
waiting_times_df = pd.read_csv(DATA_DIR / "weather_data.csv")

attendance_df = pd.read_csv(DATA_DIR / "attendance.csv")

weather_df = pd.read_csv(DATA_DIR / "weather_data.csv")

In [3]:
train_df = create_train_df(data_dir=DATA_DIR)
print('shape:', train_df.shape)
print('columns:', train_df.columns.tolist())
train_df.head()

shape: (294136, 17)
columns: ['date_hour', 'ENTITY_DESCRIPTION_SHORT', 'wait_time_avg', 'hour', 'dow', 'month', 'attendance', 'temp', 'pressure', 'humidity', 'wind_speed', 'clouds_all', 'rain_1h', 'guests_sum', 'availability', 'utilization', 'covid']


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,hour,dow,month,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,guests_sum,availability,utilization,covid
0,2018-07-21 09:00:00,Bumper Cars,5.0,9,5,7,59066.0,20.59,1014,68,3.43,62,0.0,564.000,1.0,0.553484,0
1,2018-07-21 10:00:00,Bumper Cars,5.0,10,5,7,59066.0,22.28,1014,62,2.41,66,0.0,841.999,1.0,0.826299,0
2,2018-07-21 11:00:00,Bumper Cars,12.5,11,5,7,59066.0,23.20,1014,58,2.39,59,0.0,860.002,1.0,0.843967,0
3,2018-07-21 12:00:00,Bumper Cars,17.5,12,5,7,59066.0,24.70,1014,56,2.10,34,0.0,880.004,1.0,0.863596,0
4,2018-07-21 13:00:00,Bumper Cars,12.5,13,5,7,59066.0,24.71,1014,54,2.27,51,0.0,799.996,1.0,0.785079,0


## Inference

In [4]:
attraction_name = str(train_df["ENTITY_DESCRIPTION_SHORT"].iloc[0])

used_weather_cols = [
    "dt_iso",
    "temp", "pressure", "humidity", "wind_speed",
    "clouds_all", "rain_1h", "snow_1h", "visibility",
]

weather_forecast_df = (
    pd.read_csv(DATA_DIR / "weather_data.csv", usecols=lambda c: c in used_weather_cols)
    .tail(24 * 7)
    .copy()
)

attendance_forecast_df = (
    pd.read_csv(DATA_DIR / "attendance.csv")
    .query("FACILITY_NAME == 'PortAventura World'")
    .assign(date=lambda d: pd.to_datetime(d["USAGE_DATE"], errors="coerce").dt.floor("D"))
    [["date", "attendance"]]
    .dropna()
    .drop_duplicates("date")
    .tail(7)
)

prev_week_real_df = (
    train_df[train_df["ENTITY_DESCRIPTION_SHORT"].astype(str) == attraction_name]
    [["date_hour", "guests_sum", "availability", "utilization"]]
    .tail(24 * 7)
)

print(
    "attraction:", attraction_name,
    "| weather rows:", len(weather_forecast_df),
    "| attendance days:", len(attendance_forecast_df),
    "| previous-week rows:", len(prev_week_real_df),
)


attraction: Bumper Cars | weather rows: 168 | attendance days: 7 | previous-week rows: 168


In [5]:
inference_df = create_inference_df(
    weather_forecast_df=weather_forecast_df,
    attendance_forecast_df=attendance_forecast_df,
    attraction_name=attraction_name,
    previous_week_real_df=prev_week_real_df,
)

print('shape:', inference_df.shape)
inference_df.head()

shape: (98, 17)


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,hour,dow,month,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,guests_sum,availability,utilization,covid
0,2022-08-17 09:00:00,Bumper Cars,NaN,9,2,8,0.0,18.74,1011,63,3.50,100,0.0,687.25,1.0,0.675172,0
1,2022-08-17 10:00:00,Bumper Cars,NaN,10,2,8,0.0,18.80,1011,63,3.50,100,0.0,687.25,1.0,0.675172,0
2,2022-08-17 11:00:00,Bumper Cars,NaN,11,2,8,0.0,20.50,1010,89,2.88,3,0.0,687.25,1.0,0.675172,0
3,2022-08-17 12:00:00,Bumper Cars,NaN,12,2,8,0.0,22.78,1010,78,2.88,3,0.0,687.25,1.0,0.675172,0
4,2022-08-17 13:00:00,Bumper Cars,NaN,13,2,8,0.0,24.95,1010,67,2.88,3,0.0,687.25,1.0,0.675172,0
